# Thesis Visualizations

This notebook creates the figures used in the thesis. It uses the saved benchmark outputs and the implemented system architecture.



## Setup

The figures are built with Plotly. Static image export needs Kaleido.

The notebook uses the same blue and red colors as the evaluation notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT = Path('../docs/thesis_figures')
summary = pd.read_csv('../docs/evaluation_outputs/summary_metrics.csv')
results = pd.read_csv('../docs/evaluation_outputs/benchmark_results.csv')
category_breakdown = pd.read_csv('../docs/evaluation_outputs/category_breakdown.csv')
failure_analysis = pd.read_csv('../docs/evaluation_outputs/failure_analysis.csv')

COLORS = {
    'dark': '#0F172A',
    'navy': '#0B2545',
    'blue': '#1D4E89',
    'mid': '#4F86C6',
    'light': '#A9C6E8',
    'pale': '#EAF3FB',
    'line': '#CBD5E1',
    'grid': '#E2E8F0',
    'red': '#B23A48',
    'pale_red': '#FDE2E5',
    'gray': '#64748B',
    'white': '#FFFFFF'
}

def export(fig, name, width=1500, height=900):
    fig.write_image(str(OUT / name), width=width, height=height, scale=2)


## Formal Metric Definitions

Most benchmark metrics are rates. A rate is calculated as the number of passed applicable cases divided by the number of applicable cases. Failed cases stay in the denominator.

Aggregation Accuracy checks the primary aggregation step. It does not include conversational follow-up correction. Follow-up correction is measured separately by Follow-Up Consistency.

Groundedness is a benchmark check for evidence and domain consistency. It is useful, but it is not a proof that hallucinations are impossible.


## Benchmark Charts

The next cell rebuilds the benchmark charts from the audited CSV files.

The charts use only saved evaluation outputs. They do not recalculate or change the benchmark results.


In [ ]:
rate = summary[summary['metric'] != 'Average Latency'].copy()
rate['percent'] = rate['value'] * 100
rate['color'] = rate['percent'].apply(lambda v: COLORS['red'] if v < 80 else COLORS['blue'])
fig = make_subplots(rows=1, cols=2, column_widths=[0.78, 0.22], specs=[[{'type': 'xy'}, {'type': 'indicator'}]], subplot_titles=('Rate metrics', 'Latency'))
fig.add_trace(go.Bar(y=rate['metric'], x=rate['percent'], orientation='h', marker_color=rate['color'], text=rate['display_value'], textposition='outside', cliponaxis=False), row=1, col=1)
latency = summary[summary['metric'] == 'Average Latency'].iloc[0]
fig.add_trace(go.Indicator(mode='number', value=float(latency['value']), number={'suffix': 's', 'font': {'size': 38, 'color': COLORS['navy']}}, title={'text': 'Mean latency'}), row=1, col=2)
fig.update_layout(title={'text': 'Benchmark Metric Breakdown', 'x': 0.5, 'font': {'size': 28, 'color': COLORS['dark']}}, width=1500, height=850, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 260, 'r': 80, 't': 110, 'b': 70}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']}, showlegend=False)
fig.update_xaxes(range=[0, 108], title='Percent', gridcolor=COLORS['grid'], row=1, col=1)
fig.update_yaxes(autorange='reversed', row=1, col=1)
export(fig, 'metric_breakdown.png', 1500, 850)

pass_count = int(results['analytical_correct'].sum())
fail_count = int((~results['analytical_correct'].astype(bool)).sum())
fig = go.Figure()
fig.add_bar(x=['Pass', 'Fail'], y=[pass_count, fail_count], marker_color=[COLORS['blue'], COLORS['red']], text=[pass_count, fail_count], textposition='outside')
fig.update_layout(title={'text': 'Analytical Correctness Pass/Fail Summary', 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}}, width=1000, height=650, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 80, 'r': 60, 't': 100, 'b': 70}, font={'family': 'Arial, sans-serif', 'size': 16, 'color': COLORS['dark']}, showlegend=False)
fig.update_yaxes(title='Cases', gridcolor=COLORS['grid'], rangemode='tozero')
fig.update_xaxes(title='Result')
export(fig, 'benchmark_pass_fail.png', 1000, 650)

category_counts = category_breakdown.sort_values('cases')
fig = go.Figure()
fig.add_bar(x=category_counts['cases'], y=category_counts['category'], orientation='h', marker_color=COLORS['blue'], text=category_counts['cases'], textposition='outside')
fig.update_layout(title={'text': 'Benchmark Category Distribution', 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}}, width=1300, height=800, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 270, 'r': 80, 't': 100, 'b': 70}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']}, showlegend=False)
fig.update_xaxes(title='Cases', dtick=1, gridcolor=COLORS['grid'])
fig.update_yaxes(title='')
export(fig, 'benchmark_category_distribution.png', 1300, 800)

category_accuracy = category_breakdown.sort_values('analytical_correct_rate')
fig = go.Figure()
fig.add_bar(x=category_accuracy['analytical_correct_rate'] * 100, y=category_accuracy['category'], orientation='h', marker_color=[COLORS['red'] if value < 0.8 else COLORS['blue'] for value in category_accuracy['analytical_correct_rate']], text=[f"{value:.0%} ({count})" for value, count in zip(category_accuracy['analytical_correct_rate'], category_accuracy['cases'])], textposition='outside', cliponaxis=False)
fig.update_layout(title={'text': 'Analytical Correctness by Category', 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}}, width=1400, height=850, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 280, 'r': 120, 't': 100, 'b': 70}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']}, showlegend=False)
fig.update_xaxes(title='Analytical correctness, %', range=[0, 112], gridcolor=COLORS['grid'])
fig.update_yaxes(title='')
export(fig, 'correctness_vs_category.png', 1400, 850)

failure_distribution = failure_analysis['failure_reason'].dropna().str.split(', ').explode().value_counts().rename_axis('failure_reason').reset_index(name='count').sort_values('count')
fig = go.Figure()
fig.add_bar(x=failure_distribution['count'], y=failure_distribution['failure_reason'], orientation='h', marker_color=COLORS['red'], text=failure_distribution['count'], textposition='outside')
fig.update_layout(title={'text': 'Observed Failure Distribution', 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}}, width=1200, height=650, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 220, 'r': 80, 't': 100, 'b': 70}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']}, showlegend=False)
fig.update_xaxes(title='Cases', dtick=1, gridcolor=COLORS['grid'])
fig.update_yaxes(title='')
export(fig, 'failure_distribution.png', 1200, 650)

fig = go.Figure()
fig.add_histogram(x=results['latency_seconds'], marker_color=COLORS['blue'], nbinsx=12)
fig.add_vline(x=results['latency_seconds'].median(), line_dash='dash', line_color=COLORS['red'])
fig.update_layout(title={'text': 'Latency Distribution', 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}}, width=1300, height=750, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 100, 'r': 80, 't': 100, 'b': 80}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']}, showlegend=False)
fig.update_xaxes(title='Latency, seconds', gridcolor=COLORS['grid'])
fig.update_yaxes(title='Cases', gridcolor=COLORS['grid'])
export(fig, 'latency_distribution.png', 1300, 750)


## Architecture Diagram Helpers

The architecture figures use grouped subsystem containers. They show the implemented frontend, backend, planning, execution, validation, artifact, and persistence parts.



In [ ]:
def base_fig(width=1500, height=900, title=None):
    fig = go.Figure()
    fig.update_xaxes(range=[0, 100], visible=False)
    fig.update_yaxes(range=[0, 100], visible=False)
    fig.update_layout(width=width, height=height, paper_bgcolor=COLORS['white'], plot_bgcolor=COLORS['white'], margin={'l': 35, 'r': 35, 't': 80 if title else 35, 'b': 35}, font={'family': 'Arial, sans-serif', 'size': 15, 'color': COLORS['dark']})
    if title:
        fig.update_layout(title={'text': title, 'x': 0.5, 'font': {'size': 26, 'color': COLORS['dark']}})
    return fig

def container(fig, x0, y0, x1, y1, label, fill='#F8FBFF'):
    fig.add_shape(type='rect', x0=x0, y0=y0, x1=x1, y1=y1, line={'color': COLORS['line'], 'width': 2}, fillcolor=fill, layer='below', xref='x', yref='y')
    fig.add_annotation(x=x0+2, y=y1+2, text=f'<b>{label}</b>', showarrow=False, xanchor='left', yanchor='bottom', font={'size': 15, 'color': COLORS['navy']})

def box(fig, x, y, w, h, text, fill=None, color=None):
    fill = fill or COLORS['white']
    color = color or COLORS['blue']
    fig.add_shape(type='rect', x0=x, y0=y, x1=x+w, y1=y+h, line={'color': color, 'width': 2.4}, fillcolor=fill, xref='x', yref='y')
    fig.add_annotation(x=x+w/2, y=y+h/2, text=text, showarrow=False, align='center', font={'size': 14, 'color': COLORS['dark']})

def arrow(fig, x0, y0, x1, y1, color=None):
    fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref='x', yref='y', axref='x', ayref='y', showarrow=True, arrowhead=3, arrowsize=1.25, arrowwidth=3, arrowcolor=color or COLORS['blue'])


## Overall Architecture

This figure shows the main system layers in one view. It groups the frontend, backend API, planning layer, deterministic execution layer, validation layer, artifact and reporting layer, and persistence layer.


In [ ]:
fig = base_fig(1800, 980, 'Implemented System Architecture')
container(fig, 3, 58, 17, 86, 'Frontend Layer', COLORS['pale'])
container(fig, 22, 58, 38, 86, 'Backend API Layer', '#F7FAFF')
container(fig, 43, 54, 59, 86, 'Product Orchestration', '#F7FAFF')
container(fig, 64, 26, 81, 86, 'Execution Paths', '#FBFDFF')
container(fig, 85, 43, 98, 86, 'Validation and Output', COLORS['pale'])
container(fig, 23, 17, 59, 45, 'Persistence Layer', '#FBFDFF')
box(fig, 6, 75, 8, 5, 'Browser<br>UI')
box(fig, 6, 64, 9, 5, 'Next.js<br>Workspace')
box(fig, 25, 76, 10, 5, 'Investigation<br>API')
box(fig, 25, 67, 10, 5, 'Data Source<br>API')
box(fig, 25, 58, 10, 5, 'Report<br>API')
box(fig, 46, 76, 10, 5, 'Run<br>Service')
box(fig, 46, 66, 10, 5, 'Dataset<br>Registry')
box(fig, 46, 56, 10, 5, 'Routing and<br>Context')
box(fig, 67, 76, 11, 5, 'Semantic Plan<br>Executor')
box(fig, 67, 66, 11, 5, 'Business KPI<br>Planner')
box(fig, 67, 56, 11, 5, 'Fallback /<br>Direct Analysis')
box(fig, 67, 46, 11, 5, 'Optional<br>DeepAgents Tools')
box(fig, 67, 34, 11, 5, 'Multi-dataset<br>Branches')
box(fig, 88, 76, 7, 5, 'Critic and<br>Validation', color=COLORS['red'])
box(fig, 88, 64, 7, 5, 'Adapter')
box(fig, 88, 52, 7, 5, 'Artifacts and<br>Findings')
box(fig, 88, 45, 7, 4, 'Reports')
box(fig, 27, 34, 10, 5, 'SQLite<br>Product Store')
box(fig, 43, 34, 10, 5, 'Runtime<br>Tables')
box(fig, 43, 23, 10, 5, 'Uploaded<br>CSV Files')
arrow(fig, 14, 77.5, 25, 78.5)
arrow(fig, 15, 66.5, 25, 69.5)
arrow(fig, 15, 65.0, 25, 60.5)
arrow(fig, 35, 78.5, 46, 78.5)
arrow(fig, 35, 69.5, 43, 36.5)
arrow(fig, 56, 78.5, 67, 78.5)
arrow(fig, 56, 68.5, 67, 68.5)
arrow(fig, 56, 58.5, 67, 58.5)
arrow(fig, 56, 58.5, 67, 48.5)
arrow(fig, 56, 68.5, 67, 36.5)
arrow(fig, 78, 78.5, 88, 78.5)
arrow(fig, 78, 68.5, 88, 78.0)
arrow(fig, 78, 58.5, 88, 77.5)
arrow(fig, 78, 48.5, 88, 77.0)
arrow(fig, 78, 36.5, 88, 76.5)
arrow(fig, 91.5, 76, 91.5, 69)
arrow(fig, 91.5, 64, 91.5, 57)
arrow(fig, 91.5, 52, 91.5, 49)
arrow(fig, 48, 28, 48, 34)
arrow(fig, 48, 39, 48, 66)
export(fig, 'overall_system_architecture.png', 1800, 980)

## Additional Workflow Diagrams

This cell rebuilds the workflow diagrams used in the thesis. These diagrams describe the evaluated analytical pipeline, dataset upload flow, and evaluation process.


In [ ]:
diagram_specs = [
    ('analytical_execution_pipeline.png', 'Analytical Execution Pipeline', 1700, 820, [(3, 58, 18, 86, 'User Input', COLORS['pale']), (23, 52, 42, 86, 'Run Preparation', '#F7FAFF'), (47, 52, 64, 86, 'Planning', '#F7FAFF'), (47, 24, 73, 47, 'Execution', '#FBFDFF'), (78, 34, 97, 86, 'Validated Output', COLORS['pale'])], [(6, 72, 9, 6, 'User<br>Question'), (26, 76, 12, 5, 'Run<br>Creation'), (26, 66, 12, 5, 'Conversation<br>Context'), (26, 56, 12, 5, 'Dataset<br>Resolution'), (50, 76, 10, 5, 'Routing'), (50, 66, 10, 5, 'Semantic<br>Planning'), (50, 57, 10, 5, 'Constraint<br>Checks'), (50, 38, 12, 5, 'Deterministic<br>Execution'), (50, 29, 12, 5, 'Computed<br>Evidence'), (82, 76, 10, 5, 'Critic and<br>Validation'), (82, 64, 10, 5, 'Adapter'), (82, 52, 10, 5, 'Artifacts and<br>Findings'), (82, 40, 10, 5, 'Saved<br>Response')], [], [(15, 75, 26, 78.5), (38, 78.5, 50, 78.5), (38, 68.5, 50, 68.5), (38, 58.5, 50, 59.5), (55, 57, 56, 43), (56, 38, 56, 34), (62, 31.5, 82, 78.5), (87, 76, 87, 69), (87, 64, 87, 57), (87, 52, 87, 45)], []),
    ('multi_dataset_branching.png', 'Multi-Dataset Branching', 1650, 850, [(4, 56, 22, 87, 'Linked Datasets', COLORS['pale']), (27, 56, 45, 88, 'Dataset Registry', '#F7FAFF'), (50, 45, 74, 88, 'Isolated Branch Scopes', '#FBFDFF'), (79, 42, 97, 84, 'Comparative Output', COLORS['pale'])], [(7, 76, 11, 6, 'Dataset A'), (8, 66, 11, 6, 'Dataset B'), (7, 58, 12, 6, 'Dataset C'), (30, 78, 12, 6, 'Profiles'), (30, 68, 12, 6, 'Scope<br>Scoring'), (30, 58, 12, 6, 'Selected<br>IDs'), (52, 78, 9, 5, 'Branch A<br>Run'), (52, 67, 9, 5, 'Branch B<br>Run'), (52, 56, 9, 5, 'Branch C<br>Run'), (63, 78, 9, 5, 'Evidence<br>A'), (63, 67, 9, 5, 'Evidence<br>B'), (63, 56, 9, 5, 'Evidence<br>C'), (82, 72, 10, 6, 'Comparative<br>Synthesis'), (82, 58, 10, 6, 'Validation'), (82, 46, 10, 6, 'Final<br>Answer')], [], [(18, 79, 30, 81), (19, 69, 30, 71), (19, 61, 30, 61), (42, 61, 52, 80.5), (42, 61, 52, 69.5), (42, 61, 52, 58.5), (61, 80.5, 63, 80.5), (61, 69.5, 63, 69.5), (61, 58.5, 63, 58.5), (72, 80.5, 82, 75), (72, 69.5, 82, 75), (72, 58.5, 82, 75), (87, 72, 87, 64), (87, 58, 87, 52)], []),
    ('dataset_upload_lifecycle.png', 'Dataset Upload Lifecycle', 1500, 820, [(4, 54, 24, 84, 'Frontend', COLORS['pale']), (29, 54, 48, 84, 'FastAPI Upload', '#F7FAFF'), (53, 28, 73, 84, 'Preparation', '#FBFDFF'), (78, 28, 96, 84, 'Stored Data Source', COLORS['pale'])], [(7, 72, 13, 6, 'CSV Upload<br>Form'), (9, 60, 12, 6, 'Upload Proxy<br>Size Check'), (32, 70, 12, 6, 'Multipart<br>Request'), (32, 60, 13, 6, 'CSV and Empty<br>File Checks'), (56, 72, 12, 6, 'File<br>Storage'), (58, 58, 12, 6, 'Profiling'), (56, 43, 14, 6, 'Runtime<br>SQLite Table'), (81, 70, 11, 6, 'Data Source<br>Record'), (82, 55, 11, 6, 'Profile<br>Metadata'), (81, 40, 12, 6, 'Investigation<br>Link')], [], [(19, 75, 32, 73), (20, 63, 32, 63), (44, 70, 56, 75), (44, 60, 58, 61), (64, 58, 63, 49), (70, 46, 81, 43), (68, 75, 81, 73), (68, 61, 82, 58)], []),
    ('evaluation_pipeline.png', 'Deterministic Evaluation Pipeline', 1500, 770, [(4, 58, 23, 84, 'Benchmark Input', COLORS['pale']), (29, 50, 51, 86, 'System Run', '#F7FAFF'), (57, 35, 77, 86, 'Validation', '#FBFDFF'), (82, 44, 96, 82, 'Outputs', COLORS['pale'])], [(7, 72, 12, 6, 'Benchmark<br>Case'), (8, 62, 13, 6, 'Fixed<br>Dataframe'), (32, 76, 12, 6, 'Dataset<br>Scope'), (33, 64, 12, 6, 'Planner'), (33, 53, 13, 6, 'Deterministic<br>Execution'), (60, 75, 12, 6, 'Pandas<br>Reference'), (61, 61, 12, 6, 'Metric<br>Checks'), (60, 48, 13, 6, 'Failure<br>Recording'), (84, 69, 10, 6, 'CSV<br>Outputs'), (84, 56, 10, 6, 'Figures'), (84, 45, 10, 6, 'Report<br>Tables')], [], [(19, 75, 32, 79), (20, 64, 33, 56), (39, 76, 39, 70), (39, 64, 39, 59), (46, 56, 61, 64), (66, 75, 66, 67), (73, 64, 84, 72), (73, 51, 84, 48)], [])
]

for name, title, width, height, containers, boxes, red_boxes, arrows, red_arrows in diagram_specs:
    fig = base_fig(width, height, title)
    for item in containers:
        container(fig, *item)
    for item in boxes:
        box(fig, *item)
    for item in red_boxes:
        box(fig, *item, color=COLORS['red'])
    for item in arrows:
        arrow(fig, *item)
    for item in red_arrows:
        arrow(fig, *item, COLORS['red'])
    export(fig, name, width, height)

## KPI and Multi-Dataset Support Diagrams

This cell rebuilds diagrams for KPI reasoning, follow-up correction, dataset scoring, and branch-scoped execution.


In [ ]:
additional_specs = [
    ('kpi_reasoning_flow.png', 'Business KPI Reasoning Flow', [(4, 58, 24, 84, 'Question', COLORS['pale']), (29, 48, 49, 86, 'Business Semantics', '#F7FAFF'), (54, 26, 73, 86, 'Deterministic KPI Logic', '#FBFDFF'), (79, 37, 96, 82, 'Evidence Output', COLORS['pale'])], [(8, 70, 12, 6, 'Business<br>Question'), (32, 75, 12, 6, 'Intent<br>Detection'), (33, 62, 13, 6, 'Metric<br>Mapping'), (32, 51, 14, 6, 'Constraint<br>Locking'), (57, 74, 12, 6, 'KPI<br>Registry'), (58, 59, 12, 6, 'Formula<br>Execution'), (57, 42, 14, 6, 'Grouped KPI<br>Rows'), (82, 70, 10, 6, 'Validation'), (82, 55, 11, 6, 'Artifacts'), (82, 42, 11, 6, 'Grounded<br>Synthesis')], [], [(20, 73, 32, 78), (44, 75, 57, 77), (45, 64, 58, 62), (45, 53, 58, 46), (64, 74, 64, 65), (64, 59, 64, 48), (88, 70, 88, 61), (88, 55, 88, 48)], []),
    ('followup_correction_flow.png', 'Follow-Up Correction Flow', [(4, 56, 24, 84, 'Conversation', COLORS['pale']), (29, 48, 52, 86, 'Context Recovery', '#F7FAFF'), (57, 43, 75, 86, 'Patch Attempt', '#FBFDFF'), (80, 43, 96, 82, 'Audited Result', COLORS['pale_red'])], [(7, 72, 13, 6, 'Follow-up<br>Question'), (32, 76, 13, 6, 'Recent Plan<br>Metadata'), (33, 62, 12, 6, 'Artifact<br>Context'), (33, 51, 13, 6, 'Explicit<br>Override'), (60, 74, 11, 6, 'Patch<br>Logic'), (61, 60, 12, 6, 'Re-execution'), (60, 47, 13, 6, 'New<br>Artifacts'), (83, 68, 10, 6, 'Validation'), (83, 53, 11, 7, 'Partial:<br>1 of 3 passed')], [], [(20, 75, 32, 79), (44, 76, 60, 77), (45, 64, 60, 63), (45, 54, 60, 50), (66, 74, 67, 66), (67, 60, 67, 53), (73, 50, 83, 71), (88, 68, 88, 60)], []),
    ('dataset_resolution_scoring.png', 'Dataset Resolution and Scoring', [(4, 56, 23, 84, 'Linked Datasets', COLORS['pale']), (29, 39, 51, 86, 'Registry Signals', '#F7FAFF'), (57, 45, 75, 84, 'Scoring', '#FBFDFF'), (80, 45, 96, 82, 'Decision', COLORS['pale'])], [(7, 73, 12, 6, 'Columns'), (8, 63, 12, 6, 'Samples'), (32, 76, 14, 6, 'Semantic<br>Roles'), (33, 64, 13, 6, 'Runtime<br>Availability'), (32, 52, 14, 6, 'Lineage<br>Metadata'), (60, 72, 11, 6, 'Mention<br>Match'), (61, 59, 12, 6, 'Metric and<br>Role Match'), (60, 48, 13, 6, 'Sample<br>Value Match'), (83, 70, 10, 6, 'Single<br>Dataset'), (83, 58, 10, 6, 'Several<br>Datasets'), (83, 47, 10, 6, 'Clarify')], [], [(19, 74, 32, 79), (20, 65, 33, 67), (46, 77, 60, 75), (46, 64, 61, 62), (46, 54, 60, 51), (71, 75, 83, 73), (72, 62, 83, 61), (72, 51, 83, 50)], []),
    ('branch_execution_scopes.png', 'Dataset-Scoped Execution Branches', [(4, 55, 22, 82, 'Selected Datasets', COLORS['pale']), (27, 19, 50, 86, 'Branch A Scope', '#F7FAFF'), (53, 19, 76, 86, 'Branch B Scope', '#FBFDFF'), (82, 32, 96, 82, 'Comparative Output', COLORS['pale'])], [(7, 70, 11, 6, 'Dataset A'), (7, 59, 11, 6, 'Dataset B'), (31, 75, 12, 5, 'Local<br>Context'), (31, 61, 12, 5, 'Execution'), (31, 47, 12, 5, 'Findings'), (31, 33, 12, 5, 'Artifacts'), (57, 75, 12, 5, 'Local<br>Context'), (57, 61, 12, 5, 'Execution'), (57, 47, 12, 5, 'Findings'), (57, 33, 12, 5, 'Artifacts'), (84, 68, 10, 6, 'Evidence<br>Packages'), (84, 51, 10, 6, 'Synthesis'), (84, 38, 10, 6, 'Response')], [], [(18, 73, 31, 77.5), (18, 62, 57, 77.5), (37, 75, 37, 66), (37, 61, 37, 52), (37, 47, 37, 38), (63, 75, 63, 66), (63, 61, 63, 52), (63, 47, 63, 38), (43, 35.5, 84, 71), (69, 35.5, 84, 71), (89, 68, 89, 57), (89, 51, 89, 44)], [])
]

for name, title, containers, boxes, red_boxes, arrows, red_arrows in additional_specs:
    fig = base_fig(1500, 820, title)
    for item in containers:
        container(fig, *item)
    for item in boxes:
        box(fig, *item)
    for item in red_boxes:
        box(fig, *item, color=COLORS['red'])
    for item in arrows:
        arrow(fig, *item)
    for item in red_arrows:
        arrow(fig, *item, COLORS['red'])
    export(fig, name, 1500, 820)